In [2]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

BASE_DIR = "Scientific_Novelty_Detection_2022_2025"
TRIPLET_DIR = os.path.join(BASE_DIR, "Triplets", "Novel_Papers")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
model_name = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

C:\Users\spars\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31090, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [4]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [5]:
def compute_batch_embeddings(texts, batch_size=32):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=256
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            model_output = model(**encoded)

        pooled = mean_pooling(model_output, encoded['attention_mask'])
        embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0)

In [6]:
def compute_weights_for_file(csv_path):

    df = pd.read_csv(csv_path)

    # Build texts
    pred_texts = df["pred"].fillna("").tolist()
    sub_obj_texts = (df["sub"].fillna("") + " " + df["obj"].fillna("")).tolist()

    print("Computing embeddings for predicates...")
    pred_emb = compute_batch_embeddings(pred_texts)

    print("Computing embeddings for subject-object...")
    so_emb = compute_batch_embeddings(sub_obj_texts)

    # Cosine similarity (vectorized)
    cosine_scores = torch.nn.functional.cosine_similarity(
        pred_emb, so_emb, dim=1
    )

    df["pred_weights"] = cosine_scores.numpy()

    # Save weighted file
    weighted_path = csv_path.replace("_results.csv", "_weights.csv")
    df.to_csv(weighted_path, index=False)

    print("Saved:", weighted_path)

In [7]:
TASKS = [
    "Dia2022_2025",
    "MT2022_2025",
    "QA2022_2025",
    "SA2022_2025",
    "Sum2022_2025"
]

for task in TASKS:
    path = os.path.join(TRIPLET_DIR, f"{task}_triplets_results.csv")

    if os.path.exists(path):
        print(f"\nProcessing {task}")
        compute_weights_for_file(path)
    else:
        print(f"{task} file not found.")


Processing Dia2022_2025
Computing embeddings for predicates...


100%|██████████| 556/556 [00:09<00:00, 58.68it/s]


Computing embeddings for subject-object...


100%|██████████| 556/556 [00:09<00:00, 55.61it/s]


Saved: Scientific_Novelty_Detection_2022_2025\Triplets\Novel_Papers\Dia2022_2025_triplets_weights.csv

Processing MT2022_2025
Computing embeddings for predicates...


...

Saved: Scientific_Novelty_Detection_2022_2025\Triplets\Novel_Papers\SA2022_2025_triplets_weights.csv

Processing Sum2022_2025
Computing embeddings for predicates...


100%|██████████| 19/19 [00:00<00:00, 58.89it/s]


Computing embeddings for subject-object...


100%|██████████| 19/19 [00:00<00:00, 51.69it/s]

Saved: Scientific_Novelty_Detection_2022_2025\Triplets\Novel_Papers\Sum2022_2025_triplets_weights.csv
